In [ ]:
import sys
sys.path.append("..")

from src import config
from src.factors.utils import get_latest_fundamental_period, get_latest_price_date, load_file
from src.factors import build_momentum_table, build_quality_table

import pandas as pd


In [9]:
universe = pd.read_csv("../data/raw/universe.csv", index_col="Ticker")

In [ ]:
momentum_table = []
quality_table = []
values_table = []

missing_files = []
mismatched_periods = []
missing_dates = []

for ticker in universe["Ticker"]:
    balance_sheet = load_file(ticker, missing_files, "balance sheet", "../../data/raw/fundamentals/{ticker}/balance_sheet.csv")
    income_statement = load_file(ticker, missing_files, "income statement", "../../data/raw/fundamentals/{ticker}/income_statement.csv")
    cash_flow = load_file(ticker, missing_files, "cash flow", "../../data/raw/fundamentals/{ticker}/cash_flow.csv")
    prices = load_file(ticker, missing_files, "prices", "../../data/raw/prices/{ticker}.csv")

    if any(file is None for file in [balance_sheet, income_statement, cash_flow, prices]):
        continue

    ticker_country = universe.loc[ticker, "Country"]
    
    for date in config.REBALANCE_DATES:
        latest_balance_sheet_period = get_latest_fundamental_period(balance_sheet, date)
        latest_income_statement_period = get_latest_fundamental_period(income_statement, date)
        latest_cash_flow_period = get_latest_fundamental_period(cash_flow, date)
        latest_price_date = get_latest_price_date(prices, date)

        periods = {
            "latest_balance_sheet_period": latest_balance_sheet_period, 
            "latest_income_statement_period": latest_income_statement_period, 
            "latest_cash_flow_period": latest_cash_flow_period
        }

        if len(set(periods.values())) != 1:
            mismatched_periods.append({
                "ticker": ticker,
                "date": date,
                "balance_sheet_period": latest_balance_sheet_period,
                "income_statement_period": latest_income_statement_period,
                "cash_flow_period": latest_cash_flow_period
            })

        periods_missing = [k for k, v in periods.items() if v is None]
        if len(periods_missing) > 0:
            missing_dates.append({
                "ticker": ticker,
                "date": date,
                "missing_periods": periods_missing,
            })

        build_quality_table(ticker, ticker_country, date, balance_sheet, income_statement, latest_balance_sheet_period, latest_income_statement_period, quality_table)

In [ ]:
momentum_table, missing_prices_log, nan_log = build_momentum_table(universe)
print(missing_prices_log)
print(nan_log)

Empty DataFrame
Columns: []
Index: []
     ticker       date         metric
0     REL.L 2023-01-01  momentum_12_1
1    EXPN.L 2023-01-01  momentum_12_1
2    AUTO.L 2023-01-01  momentum_12_1
3     RMV.L 2023-01-01  momentum_12_1
4      RM.L 2023-01-01  momentum_12_1
..      ...        ...            ...
134    CTAS 2023-01-01  momentum_12_1
135    JKHY 2023-01-01  momentum_12_1
136     ROL 2023-01-01  momentum_12_1
137     UNF 2023-01-01  momentum_12_1
138    HCSG 2023-01-01  momentum_12_1

[139 rows x 3 columns]


In [11]:
balance_sheet = pd.read_csv("../data/raw/fundamentals/WTS/balance_sheet.csv", index_col=0)
print(balance_sheet.loc["Total Debt", "2025-12-31"])
# print(universe.index)
# print(universe.loc["WTS", "Country"])

197700000.0
